# 第二讲 — Julia 与值函数迭代 (VFI)

**异质性主体宏观经济学的计算方法**  
孙杰

用值函数迭代 (Value Function Iteration, VFI) 求解无收入冲击下的消费-储蓄问题。

步骤：
1. 将 $V$ 定义为有限财富网格上的向量。
2. 将 Bellman 算子 $V \mapsto \mathcal{T}V$ 定义为一个把向量映射为向量的函数。
3. 反复应用 $\mathcal{T}\mathcal{T}\cdots\mathcal{T}\mathcal{T}V$ 直至收敛。

说明：
- 这份代码并未做太多优化。我使用了会分配内存的函数、朴素的网格搜索、没有用 Howard 改进等等。我写它的目的是让代码清晰，而不是追求速度。


### 环境配置


In [ ]:
using Pkg
Pkg.activate("..")
Pkg.instantiate()
using Plots

# 核心代码


### 效用函数与对数网格

在 `c > 0` 上使用 CRRA 效用函数，`γ = 1` 的情形归约为 `log`。
对数间隔网格 (log-spaced grid) 将网格点集中在借贷约束附近。


In [ ]:
u(c) = c <= 0 ? -Inf : log(c)
loggrid(lo, hi, N) = exp.(range(log(lo), log(hi); length=N))

### Bellman 算子

给定一个值函数（猜测）$V$，对每个 $b$ 计算
$$
TV(b) = \max_{b'} u(c) + \beta V(b')
$$
其中
$$
\begin{align*}
    c &= b - b^\text{end}\\
    b^\text{end} &= \frac{b' - z}{R}
\end{align*}
$$

$$
TV(b) = \max_{b'} u(c) + \beta V(b'),
$$
$$
\begin{align*}
    c &= b - b^\text{end},\\
    b^\text{end} &= \frac{b' - z}{R}.
\end{align*}
$$

这里 $z$ 是个体的生产率 / 人力资本水平；在第二讲中它是一个确定性的标量，到第三讲中将变为随机的。


In [ ]:
"""
应用 Bellman 算子 V |-> TV，其中 V 是定义在财富网格上的向量，且不存在收入冲击。

"""
function bellman_operator(V, params)
    # 从 `params` 中解包参数
    (; β, R, z, b_grid) = params

    # 计算实现每个下期财富水平 `b_next` 所需的储蓄水平 `b_end`
    b_end = (b_grid' .- z)./R

    return maximum(u.(b_grid .- b_end) .+ β.*V'; dims=2)
end

### 值函数迭代

反复迭代 $V \mapsto \mathcal{T}V$，直到最大绝对变化小于 `tol`。


In [ ]:
function solve_vfi(params; tol=1e-6, maxiter=2000, verbosity=0)
    # 给出 V 的初始猜测
    V = zeros(size(params.b_grid))

    Δ = Inf
    iters = 0
    while Δ >= tol
        # 更新 V
        TV = bellman_operator(V, params)

        # 计算误差
        Δ = maximum(abs.(TV .- V))

        # 更新 V
        V = TV
        iters = iters + 1

        # 若 iters > maxiter，则报错。
        iters > maxiter && error("VFI did not converge in $maxiter iterations")
    end

    # 若 verbosity >= 1，则打印收敛信息。
    verbosity >= 1 && println("VFI converged in $iters iterations with error $Δ")

    return V
end

In [ ]:
b_grid = loggrid(0.05, 40.0, 400)
params = (;β=0.96, R=1.04, z=1.0, b_grid)

@show solve_vfi(params, verbosity=1)
;

## 求解与绘图

V 是单调递增且凹的。

随着迭代次数的增加，每次更新的效果越来越小。


In [ ]:
b_grid = loggrid(0.05, 40.0, 400)
params = (;β=0.96, R=1.04, z=1.0, b_grid)

# 给出 V 的初始猜测
V = zeros(size(b_grid))
V_0 = copy(V)

for i = 1:20
    V = bellman_operator(V, params)
end
V_20 = copy(V)

for i = 1:20
    V = bellman_operator(V, params)
end
V_40 = copy(V)

for i = 1:20
    V = bellman_operator(V, params)
end
V_60 = copy(V)

for i = 1:20
    V = bellman_operator(V, params)
end
V_80 = copy(V)

for i = 1:20
    V = bellman_operator(V, params)
end
V_100 = copy(V)

plot(b_grid, hcat(V_0, V_20, V_40, V_60, V_80, V_100), label=["V_0" "V_20" "V_40" "V_60" "V_80" "V_100"])

### 策略函数 (Policy Function)

策略函数与 Bellman 算子是同一个算子，只是把 `max` 换成 `argmax`：
$$
c^\star(b) \;=\; \arg\max_{c \in [0, b]} \; u(c) + \beta\, V\!\big(R(b - c) + z\big).
$$

$$
c^\star(b) \;=\; \arg\max_{c \in [0, b]} \; u(c) + \beta\, V\!\big(R(b - c) + z\big).
$$


In [ ]:
"""
从 V 计算策略索引向量 `c_ind`。对 b_grid 的每一行，
返回最优下期财富 b' 在 b_grid 中的列索引。
与 `bellman_operator` 是同一个算子，只是把 `max` 换成 `argmax`。

"""
function policy_function(V, params)
    (; β, R, z, b_grid) = params

    # 与 bellman_operator 相同的设置
    b_end = (b_grid' .- z) ./ R

    # 对 b'（列）取 argmax —— 从每一行中提取列索引
    idx = argmax(u.(b_grid .- b_end) .+ β .* V'; dims=2)
    return vec(getindex.(idx, 2))::Vector{Int}
end


In [ ]:
# 从收敛后的 V 读出策略 —— c_ind[i] 是在 b_grid[i] 处选择的 b' 的索引
c_ind = policy_function(V, params);


# 分步讲解


### 效用函数与对数网格


In [ ]:
u(c) = c <= 0 ? -Inf : log(c)
loggrid(lo, hi, N) = exp.(range(log(lo), log(hi); length=N))

In [ ]:
# 对某个消费水平计算效用，
@show u(2.5)
@show u(3.0)
# 对多个可能的消费水平计算效用，
@show u.([1, 2, 3])
# 对整个可能的消费水平网格计算效用。
@show c_grid = loggrid(0.1, 10.0, 100)
@show u.(c_grid)
;

### Bellman 算子


In [ ]:
"""
应用 Bellman 算子 V |-> TV，其中 V 是定义在财富网格上的向量，且不存在收入冲击。

"""
function bellman_operator(V, params)
    # 从 `params` 中解包参数
    (; β, R, z, b_grid) = params

    # 计算实现每个下期财富水平 `b_next` 所需的储蓄水平 `b_end`
    b_end = (b_grid' .- z)./R

    return maximum(u.(b_grid .- b_end) .+ β.*V'; dims=2)
end

In [ ]:
b_grid = loggrid(0.05, 40.0, 400)
β=0.96
R=1.04
z=1.0

# 可以把第 2 个维度（列）想成代表下期财富 `b_next`。
# 我们只需对 b_grid 做转置，就能得到 `b_next` 的网格。

# 注意，b_grid 是一个列向量
b_grid

In [ ]:
# 而 b_grid' 是一个行向量
b_grid'

In [ ]:
z = 0.5

# 那么，实现每个 `b_next` 所需的期末财富为：
@show b_end = (b_grid' .- z)./R
# 注意 `b_end` 仍然是一个行向量，因为 b_end 中的每一项都对应不同的 b_next 水平。

# 另一种思考方式：
# b_next_grid = b_grid'
# b_end = (b_next_grid .- z)./R

# 对每个当前的财富水平（行），家庭必须选择下期财富 `b_next`（列）。
# 那么各种可能的效用值就构成一个矩阵：每个可能的 b 对应一行，每个可能的 b_next 对应一列。
u.(b_grid .- b_end)

In [ ]:
# 现在我们来求解决策问题。
# 假设我们已经有了 V 的某个猜测
V = ones(size(b_grid))

# 我们可以定义 `V_next` 为每种下期财富 `b_next` 下（猜测的）取值。
# 对应每个 b_next 的 `V_next` 网格就是 V'。
# 那么 u + βV_next 的矩阵就是：
u.(b_grid .- b_end) .+ β.*V'

In [ ]:
# 家庭决策问题的解就是
maximum(u.(b_grid .- b_end) .+ β.*V'; dims=2)

In [ ]:
# 如果反复应用 Bellman 算子会发生什么？
# 设置参数
b_grid = loggrid(0.05, 40.0, 400)
params = (;β=0.96, R=1.04, z=1.0, b_grid)

# 给出 V 的猜测
V = zeros(size(b_grid))
res_mat = hcat(b_grid, V)

for i = 1:200
    V = bellman_operator(V, params)
    res_mat = hcat(res_mat, V)
end

res_mat

In [ ]:
# 初始猜测不会改变 V 收敛到的极限。
# 给出 V 的猜测
V = ones(size(b_grid))
res_mat = hcat(b_grid, V)

for i = 1:200
    V = bellman_operator(V, params)
    res_mat = hcat(res_mat, V)
end

res_mat

### 值函数迭代 (VFI)


In [ ]:
function solve_vfi(params; tol=1e-6, maxiter=2000, verbosity=0)
    # 给出 V 的初始猜测
    V = zeros(size(params.b_grid))

    Δ = Inf
    iters = 0
    while Δ >= tol
        # 更新 V
        TV = bellman_operator(V, params)

        # 计算误差
        Δ = maximum(abs.(TV .- V))

        # 更新 V
        V = TV
        iters = iters + 1

        # 若 iters > maxiter，则报错。
        iters > maxiter && error("VFI did not converge in $maxiter iterations")
    end

    # 若 verbosity >= 1，则打印收敛信息。
    verbosity >= 1 && println("VFI converged in $iters iterations with error $Δ")

    return V
end